## fetch data with the same pubmed_id as local frameintell_aux_minilmv6 from remote frameintell_pubmed.

In [1]:
from src.connection import client, local_client
from src.utils import fetch_one_day_data, fetch_data

import json
import os
import datetime
import calendar

import time
import memory_profiler

%load_ext memory_profiler

In [2]:
index_pubmed = 'frameintell_pubmed'
index_aux = 'frameintell_aux_minilmv6'

### fetch all the pubmed_id in frameintell_aux_minilmv6 from local machine

In [3]:
%%time
%%memit

query = {"_source": ["_id"], "query": {"match_all": {}}}

data = local_client.search(
        body = query,
        index = index_aux,
        size=10000,
        scroll = '10m'
    )

scroll_id = data['_scroll_id']
scroll_size = len(data['hits']['hits'])

ids = []
for item in data['hits']['hits']:
        ids.append(item['_id'].split(":")[1])
        
while scroll_size>0:
    print(scroll_size)
    data = local_client.scroll(
            scroll_id=scroll_id, 
            scroll='10m'
        )
    scroll_id = data['_scroll_id']
    scroll_size = len(data['hits']['hits'])
    
    for item in data['hits']['hits']:
        ids.append(item['_id'].split(":")[1])

print(len(ids))

10000
10000
1769
21769
peak memory: 67.54 MiB, increment: 9.23 MiB
CPU times: user 113 ms, sys: 33.4 ms, total: 146 ms
Wall time: 9.03 s


### Fetch data with correspongding ids from frameintell_pubmed of remote opensearch to local machine

In [4]:
startIdx = 0
batchSize = 10000
while startIdx < len(ids):
    print(startIdx)
    if startIdx+batchSize<len(ids):
        query = {"query": {"terms": {"_id": ids[startIdx:startIdx+batchSize]}}}
    else:
        query = {"query": {"terms": {"_id": ids[startIdx:len(ids)]}}}

    data = client.search(
            body = query,
            index = index_pubmed,
            size=10000,
        )

    scroll_size = len(data['hits']['hits'])
    startIdx = startIdx+batchSize

    filename = f"data_pubmed_{startIdx}.json"
    with open(os.path.join("data/original_data", filename), "w") as f:
        f.write(json.dumps(data['hits']['hits']))

0
10000
20000


### create index in local opensearch

In [5]:
index_pubmed = "frameintell_pubmed"

# Read the mapping.json file
with open("mappings_frameintell_pubmed.json", "r") as mapping_file:
    mapping_json = mapping_file.read()
#     print(mapping_json)

try:
    response = local_client.indices.create(index_pubmed,body=mapping_json)
    print("Creating index:")
    print(response)
except Exception as e:
    print(e)

RequestError(400, 'resource_already_exists_exception', 'index [frameintell_pubmed/2UZKZ2SgSGyKfgFX59QkOQ] already exists')


### Bulk data to local opensearch

In [10]:
startIndices = [n*10000 for n in range(1,4)]
for startIdx in startIndices:
    filename = f"data_pubmed_{startIdx}.json"
    with open(os.path.join("data/original_data", filename), "r") as f:
        data = json.load(f)

    print(f"bulk data: {filename}")
    docs = []
    for item in data:
        item_index = {"index": {"_index": index_pubmed, "_id": item["_id"]}}
        item_data = item['_source']

        docs.append(item_index)
        docs.append(item_data)

    resp = local_client.bulk(body=docs, index=index_pubmed)
local_client.close()

bulk data: data_pubmed_10000.json
bulk data: data_pubmed_20000.json
bulk data: data_pubmed_30000.json


In [11]:
client.close()
local_client.close()